# Chocolate Sales Analysis and Profit Prediction

**Objective:** Use Python and Machine Learning to analyze sales data, visualize insights, and predict profit.

Charts included: Bar Chart, Line Chart, Pie Chart, Scatter Plot, and Histogram.

## Step 1: Import required libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score

plt.rcParams['figure.figsize'] = (10, 6)

## Step 2: Load the dataset

Keep all CSV files in the same folder as this notebook. If they are in another folder, update `DATA_PATH`.

In [ ]:
DATA_PATH = "./"

sales = pd.read_csv(DATA_PATH + "sales.csv", parse_dates=["order_date"])
products = pd.read_csv(DATA_PATH + "products.csv")
customers = pd.read_csv(DATA_PATH + "customers.csv")
stores = pd.read_csv(DATA_PATH + "stores.csv")
calendar = pd.read_csv(DATA_PATH + "calendar.csv")

print("Sales:", sales.shape)
print("Products:", products.shape)
print("Customers:", customers.shape)
print("Stores:", stores.shape)
print("Calendar:", calendar.shape)

## Step 3: Understand the data

In [ ]:
sales.head()

In [ ]:
sales.info()

In [ ]:
sales.isnull().sum()

## Step 4: Merge the files into one analysis table

In [ ]:
df = sales.merge(products, on="product_id", how="left")
df = df.merge(stores, on="store_id", how="left")
df = df.merge(customers, on="customer_id", how="left")

# Create time-based columns
df["month"] = df["order_date"].dt.month
df["year_month"] = df["order_date"].dt.to_period("M").astype(str)
df["day_of_week"] = df["order_date"].dt.dayofweek

print(df.shape)
df.head()

## Step 5: Basic business summary

In [ ]:
print("Total Revenue:", round(df["revenue"].sum(), 2))
print("Total Profit:", round(df["profit"].sum(), 2))
print("Total Orders:", df["order_id"].nunique())
print("Average Profit per Order:", round(df["profit"].mean(), 2))

## Step 6: Bar Chart — Profit by product category

In [ ]:
category_profit = df.groupby("category")["profit"].sum().sort_values(ascending=False)

plt.figure()
category_profit.plot(kind="bar")
plt.title("Profit by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Total Profit")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Step 7: Line Chart — Monthly revenue trend

In [ ]:
monthly_revenue = df.groupby("year_month")["revenue"].sum()

plt.figure()
monthly_revenue.plot(kind="line", marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Step 8: Pie Chart — Revenue share by country

In [ ]:
country_revenue = df.groupby("country")["revenue"].sum().sort_values(ascending=False)

plt.figure()
country_revenue.plot(kind="pie", autopct="%1.1f%%", startangle=90)
plt.title("Revenue Share by Country")
plt.ylabel("")
plt.tight_layout()
plt.show()

## Step 9: Scatter Plot — Revenue vs Profit

In [ ]:
sample_df = df.sample(5000, random_state=42)

plt.figure()
plt.scatter(sample_df["revenue"], sample_df["profit"], alpha=0.4)
plt.title("Revenue vs Profit")
plt.xlabel("Revenue")
plt.ylabel("Profit")
plt.tight_layout()
plt.show()

## Step 10: Histogram — Distribution of customer age

In [ ]:
plt.figure()
df["age"].dropna().plot(kind="hist", bins=20)
plt.title("Customer Age Distribution")
plt.xlabel("Age")
plt.ylabel("Number of Customers / Sales Records")
plt.tight_layout()
plt.show()

## Step 11: Prepare data for Machine Learning

Target variable: `profit`  
Goal: Predict profit using sales, product, customer, store, and date features.

In [ ]:
features = [
    "quantity", "unit_price", "discount", "cocoa_percent", "weight_g",
    "age", "loyalty_member", "month", "day_of_week",
    "category", "brand", "country", "store_type", "gender"
]

target = "profit"

ml_df = df[features + [target]].dropna()

# Use a sample to make the model train faster on normal laptops
ml_df = ml_df.sample(n=min(100000, len(ml_df)), random_state=42)

X = ml_df[features]
y = ml_df[target]

X.head()

## Step 12: Train-test split and preprocessing

In [ ]:
categorical_features = ["category", "brand", "country", "store_type", "gender"]
numerical_features = [col for col in features if col not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", DecisionTreeRegressor(max_depth=10, random_state=42))
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Step 13: Train the Machine Learning model

In [ ]:
model.fit(X_train, y_train)
print("Model training completed")

## Step 14: Evaluate the model

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error:", round(mae, 2))
print("R2 Score:", round(r2, 4))

## Step 15: Actual vs predicted profit visualization

In [ ]:
results = pd.DataFrame({"Actual Profit": y_test.values, "Predicted Profit": y_pred})
results_sample = results.sample(100, random_state=42).reset_index(drop=True)

plt.figure()
plt.plot(results_sample["Actual Profit"], label="Actual Profit")
plt.plot(results_sample["Predicted Profit"], label="Predicted Profit")
plt.title("Actual vs Predicted Profit")
plt.xlabel("Sample Index")
plt.ylabel("Profit")
plt.legend()
plt.tight_layout()
plt.show()

## Final conclusion

Write your project conclusion like this:

- The project analyzed chocolate sales data across products, customers, stores, and countries.
- Bar, line, pie, scatter, and histogram charts were used to identify sales and profit patterns.
- Product category, store type, country, quantity, price, and discount were important for understanding profit.
- A Decision Tree Regression model was built to predict profit from business features.
- The project demonstrates data analysis, visualization, feature engineering, and machine learning skills.